# M5 Forecasting — Exploratory Data Analysis

**Project:** Retail Demand Forecasting & Intelligent Inventory Planning

This notebook performs the first exploratory analysis of the validated M5 dataset.

### Analysis levels

1. Overall demand
2. Store/category demand
3. Individual item-store series

We deliberately avoid immediately reshaping all 30,490 series into a massive long-format table. The raw M5 sales table contains 1,913 historical demand observations per series, so we first perform efficient aggregate and representative-series analysis.

### Questions

- What is the overall demand trend?
- What weekly/monthly seasonality exists?
- How different are stores and categories?
- How much demand is zero/intermittent?
- Which series are representative for later forecasting experiments?
- Are promotions, events, and prices associated with demand changes?


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()

# If the notebook is launched from notebooks/, move one level up.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DIR)


In [ ]:
# Load the calendar table.
calendar = pd.read_csv(
    RAW_DIR / "calendar.csv",
    parse_dates=["date"]
)

calendar.head()


In [ ]:
# Load the sales table.
# This is a wide table: one row per item-store series and one column per day.
sales = pd.read_csv(RAW_DIR / "sales_train_validation.csv")

print("Shape:", sales.shape)
print("Memory usage:", f"{sales.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
sales.head()


In [ ]:
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in sales.columns if c.startswith("d_")]

print("ID columns:", id_cols)
print("Number of day columns:", len(day_cols))
print("First day:", day_cols[0])
print("Last day:", day_cols[-1])


## 1. Dataset overview

In [ ]:
overview = pd.DataFrame({
    "metric": [
        "Number of series",
        "Number of historical days",
        "Number of stores",
        "Number of items",
        "Number of departments",
        "Number of categories",
        "Number of states",
    ],
    "value": [
        len(sales),
        len(day_cols),
        sales["store_id"].nunique(),
        sales["item_id"].nunique(),
        sales["dept_id"].nunique(),
        sales["cat_id"].nunique(),
        sales["state_id"].nunique(),
    ]
})

overview


## 2. Aggregate daily demand

In [ ]:
# Sum demand across all 30,490 bottom-level series.
daily_total = sales[day_cols].sum(axis=0)

daily = calendar[["d", "date", "wm_yr_wk", "weekday", "wday", "month", "year",
                   "event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI"]].copy()

daily["sales"] = daily["d"].map(daily_total)

daily.head()


In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(daily["date"], daily["sales"])
ax.set_title("Total Daily Demand Across All M5 Series")
ax.set_xlabel("Date")
ax.set_ylabel("Units Sold")
plt.tight_layout()
plt.show()


## 3. Weekly seasonality

In [ ]:
weekday_order = [
    "Saturday", "Sunday", "Monday", "Tuesday",
    "Wednesday", "Thursday", "Friday"
]

weekday_summary = (
    daily.groupby("weekday", observed=True)["sales"]
    .agg(["mean", "median", "std"])
    .reindex(weekday_order)
)

weekday_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
weekday_summary["mean"].plot(kind="bar", ax=ax)
ax.set_title("Average Daily Demand by Weekday")
ax.set_xlabel("Weekday")
ax.set_ylabel("Average Units Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Monthly seasonality

In [ ]:
monthly_summary = (
    daily.groupby("month", observed=True)["sales"]
    .agg(["mean", "median", "std"])
)

monthly_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
monthly_summary["mean"].plot(kind="bar", ax=ax)
ax.set_title("Average Daily Demand by Month")
ax.set_xlabel("Month")
ax.set_ylabel("Average Units Sold")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Store-level demand

In [ ]:
store_totals = (
    sales.groupby("store_id")[day_cols]
    .sum()
    .sum(axis=1)
    .sort_values(ascending=False)
    .rename("total_sales")
)

store_totals


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
store_totals.plot(kind="bar", ax=ax)
ax.set_title("Total Historical Demand by Store")
ax.set_xlabel("Store")
ax.set_ylabel("Units Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Category-level demand

In [ ]:
category_totals = (
    sales.groupby("cat_id")[day_cols]
    .sum()
    .sum(axis=1)
    .sort_values(ascending=False)
    .rename("total_sales")
)

category_totals


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
category_totals.plot(kind="bar", ax=ax)
ax.set_title("Total Historical Demand by Category")
ax.set_xlabel("Category")
ax.set_ylabel("Units Sold")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Zero-demand and intermittent series

In [ ]:
series_zero_counts = (sales[day_cols] == 0).sum(axis=1)

zero_demand_summary = pd.Series({
    "series": len(series_zero_counts),
    "series_with_at_least_one_zero_day": int((series_zero_counts > 0).sum()),
    "series_with_50%+ zero days": int((series_zero_counts >= len(day_cols) * 0.50).sum()),
    "completely zero series": int((series_zero_counts == len(day_cols)).sum()),
})

zero_demand_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(series_zero_counts, bins=40)
ax.set_title("Distribution of Zero-Demand Days per Series")
ax.set_xlabel("Number of zero-demand days")
ax.set_ylabel("Number of series")
plt.tight_layout()
plt.show()


## 8. Top-selling item-store series

In [ ]:
sales["total_sales"] = sales[day_cols].sum(axis=1)

top_series = (
    sales[
        ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "total_sales"]
    ]
    .sort_values("total_sales", ascending=False)
    .head(20)
)

top_series


In [ ]:
top_ids = top_series["id"].tolist()

top_sales = (
    sales.set_index("id")
    .loc[top_ids, day_cols]
    .T
)

top_sales.index = pd.to_datetime(
    calendar.set_index("d").loc[top_sales.index, "date"].values
)

fig, ax = plt.subplots(figsize=(15, 6))
for series_id in top_sales.columns[:10]:
    ax.plot(top_sales.index, top_sales[series_id], label=series_id)

ax.set_title("Demand History for Selected High-Volume Series")
ax.set_xlabel("Date")
ax.set_ylabel("Units Sold")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)
plt.tight_layout()
plt.show()


## 9. Calendar events

In [ ]:
event_days = daily[
    daily["event_name_1"].notna() | daily["event_name_1"].notnull()
].copy()

# Keep only actual named events.
event_days = daily[daily["event_name_1"].notna()]

event_days[
    ["date", "event_name_1", "event_type_1", "sales"]
].head(20)


In [ ]:
event_effect = (
    daily.assign(is_event=daily["event_name_1"].notna())
    .groupby("is_event")["sales"]
    .agg(["count", "mean", "median"])
)

event_effect


## 10. EDA findings template

After executing the notebook, record evidence-based findings here.

### Trend
- 

### Weekly seasonality
- 

### Monthly/annual seasonality
- 

### Store differences
- 

### Category differences
- 

### Intermittent demand
- 

### Events/promotions
- 

### Candidate representative series
- High-volume:
- Seasonal:
- Intermittent:
- Typical:
- Volatile:

> Do not fill these conclusions before reviewing the plots and summary statistics.


## Next step

After reviewing this EDA, we will create a **representative forecasting dataset** and a proper **time-based train/validation/test strategy**.

We will establish a seasonal-naive baseline before implementing SARIMA, Prophet, and LSTM/GRU.
